# 09 — Árboles (Random Forest, XGBoost) y tabla comparativa final (Persona B)

**Objetivo:** probar si modelos no lineales (árboles) mejoran a la logística con **las mismas variables** del
modelo final (`FEATURES_FINAL`), y armar la tabla comparativa de todos los modelos en validación (hito #9).

**Reglas que se respetan:**
- Ajuste de hiperparámetros solo con `TimeSeriesSplit(5)` **dentro de train** (2017-18 a 2022-23).
- Evaluación en validación (2023-24) sobre **los mismos 380 partidos que las cuotas**, siempre con `evaluate()`.
- El test (2024-25) no se toca.

**Criterio de decisión:** gana el menor log loss en validación; si la diferencia con el modelo más simple
no es significativa según un bootstrap pareado, se queda el más simple.

In [ ]:
import sys; sys.path.append("..")
import time
import numpy as np
import pandas as pd

from src.splits import split
from src.features.build import FEATURE_COLS
from src.models.final import FEATURES_FINAL
from src.models.baseline import FrequencyBaseline
from src.models.logistic import tune_logistic
from src.models.poisson import PoissonElo
from src.models.trees import make_random_forest, make_xgboost, tune_random_forest, tune_xgboost
from src.models.common import LABEL_A_INT, proba_hda
from src.evaluation.metrics import evaluate, compare, implied_probs
from src.evaluation.registro import guardar_resultado, RUTA_DEFECTO
from src.evaluation.bootstrap import tabla_bootstrap
from src.evaluation.figuras import fig_comparacion_modelos

df = pd.read_csv("../data/processed/features.csv", parse_dates=["date"])
train, val = split(df)
train = train.sort_values(["date", "match_id"])
assert train["date"].max() < val["date"].min()
val_ok = val[val[["odds_H", "odds_D", "odds_A"]].notna().all(axis=1)]
y = val_ok["result"]
print(f"train: {len(train)} | val con cuotas: {len(val_ok)}")
print("Variables:", FEATURES_FINAL)

## 1. El problema de las etiquetas en XGBoost

`XGBClassifier` no acepta etiquetas de texto: hay que pasarle números (`LABEL_A_INT = {"H": 0, "D": 1, "A": 2}`).
Entonces su `classes_` es `[0, 1, 2]` y `to_hda()`, que busca `"H"`, `"D"`, `"A"`, falla. `proba_hda(model, X)`
(en `src/models/common.py`) traduce los números de vuelta a letras y devuelve siempre las columnas en orden
H, D, A, sirva el modelo con etiquetas de texto (logística, Random Forest) o numéricas (XGBoost).

In [ ]:
print(LABEL_A_INT)
xgb_mini = make_xgboost(n_estimators=10, max_depth=2).fit(train[FEATURES_FINAL], train["result"].map(LABEL_A_INT))
print("classes_ de XGBoost:", list(xgb_mini.classes_))
print("Primeras filas en orden H, D, A:")
print(pd.DataFrame(proba_hda(xgb_mini, val_ok[FEATURES_FINAL].head(3)), columns=["H", "D", "A"]).round(3))

## 2. Random Forest

Pipeline: mediana para los NaN de inicio de temporada → bosque de 400 árboles. Se buscan
`max_depth` ∈ {3, 5, 8} y `min_samples_leaf` ∈ {20, 50, 100} (9 combinaciones × 5 pliegues).
Hojas grandes y árboles poco profundos = modelo más suave, menos riesgo de memorizar ruido.

In [ ]:
t0 = time.time()
busq_rf = tune_random_forest(train, FEATURES_FINAL)
print(f"Tiempo: {time.time() - t0:.0f} s | mejor: {busq_rf.best_params_} | log loss CV = {-busq_rf.best_score_:.4f}")
cv_rf = (pd.DataFrame(busq_rf.cv_results_["params"])
         .assign(log_loss_cv=-busq_rf.cv_results_["mean_test_score"])
         .pivot(index="bosque__max_depth", columns="bosque__min_samples_leaf", values="log_loss_cv"))
cv_rf.round(4)

## 3. XGBoost

`objective="multi:softprob"`, `subsample=0.8`. Se buscan `max_depth` ∈ {2, 3, 4},
`learning_rate` ∈ {0.03, 0.1} y `n_estimators` ∈ {200, 400} (12 combinaciones × 5 pliegues).
XGBoost maneja los NaN por sí mismo (aprende hacia qué rama mandarlos), así que no lleva imputador.

In [ ]:
t0 = time.time()
busq_xgb = tune_xgboost(train, FEATURES_FINAL)
print(f"Tiempo: {time.time() - t0:.0f} s | mejor: {busq_xgb.best_params_} | log loss CV = {-busq_xgb.best_score_:.4f}")
(pd.DataFrame(busq_xgb.cv_results_["params"])
   .assign(log_loss_cv=-busq_xgb.cv_results_["mean_test_score"])
   .sort_values("log_loss_cv").reset_index(drop=True).round(4))

Para comparar en la misma escala: la logística sin redundancia, con las mismas variables y los mismos pliegues.

In [ ]:
busq_log = tune_logistic(train, FEATURES_FINAL)
print(f"Logística sin redundancia | log loss CV = {-busq_log.best_score_:.4f}")
print(f"Random Forest             | log loss CV = {-busq_rf.best_score_:.4f}")
print(f"XGBoost                   | log loss CV = {-busq_xgb.best_score_:.4f}")

## 4. Sobreajuste: lo que el modelo aprende de memoria

Si un modelo tiene log loss mucho mejor en los partidos con los que se entrenó que en partidos nuevos,
está memorizando ruido (sobreajuste). Comparamos la versión elegida de cada árbol con una versión
**sin frenos** (árboles profundos / muchas rondas con tasa alta).

In [ ]:
variantes = {
    "logistica_sin_redundancia": busq_log.best_estimator_,
    "random_forest (elegido)": busq_rf.best_estimator_,
    "random_forest sin límites": make_random_forest(min_samples_leaf=1).fit(train[FEATURES_FINAL], train["result"]),
    "xgboost (elegido)": busq_xgb.best_estimator_,
    "xgboost sin frenos": make_xgboost(max_depth=4, learning_rate=0.1, n_estimators=400)
                            .fit(train[FEATURES_FINAL], train["result"].map(LABEL_A_INT)),
}
filas = []
for nombre, m in variantes.items():
    ll_train = evaluate(train["result"], proba_hda(m, train[FEATURES_FINAL]), nombre)["log_loss"]
    ll_val = evaluate(y, proba_hda(m, val_ok[FEATURES_FINAL]), nombre)["log_loss"]
    filas.append({"modelo": nombre, "log_loss_train": ll_train, "log_loss_val": ll_val,
                  "brecha": ll_val - ll_train})
pd.DataFrame(filas).round(4)

## 5. Evaluación en validación y registro

Los dos árboles elegidos se evalúan en los 380 partidos de validación con cuotas y se guardan en
`docs/resultados_validacion.csv`.

In [ ]:
res_rf = evaluate(y, proba_hda(busq_rf.best_estimator_, val_ok[FEATURES_FINAL]), "random_forest")
res_xgb = evaluate(y, proba_hda(busq_xgb.best_estimator_, val_ok[FEATURES_FINAL]), "xgboost")
guardar_resultado(res_rf)
guardar_resultado(res_xgb)
compare([res_rf, res_xgb])

## 6. Tabla comparativa final (hito #9)

La tabla sale de `docs/resultados_validacion.csv` con `compare()`. Para el bootstrap hacen falta las
probabilidades **partido a partido** de cada modelo, así que se recalculan exactamente como en los
notebooks 01, 06, 07 y 08, y se comprueba que dan las mismas cifras que el CSV.

In [ ]:
probs = {
    "baseline_frecuencias": FrequencyBaseline().fit(None, train["result"]).predict_proba(val_ok),
    "cuotas_mercado": implied_probs(val_ok),
    "poisson_elo": PoissonElo().fit(train).predict_proba(val_ok),
    "random_forest": proba_hda(busq_rf.best_estimator_, val_ok[FEATURES_FINAL]),
    "xgboost": proba_hda(busq_xgb.best_estimator_, val_ok[FEATURES_FINAL]),
}
for nombre, cols in {"logistica_elo": ["elo_diff"], "logistica_todas": FEATURE_COLS,
                     "logistica_sin_redundancia": FEATURES_FINAL}.items():
    probs[nombre] = proba_hda(tune_logistic(train, cols).best_estimator_, val_ok[cols])

registro = pd.read_csv(RUTA_DEFECTO)
for fila in registro.itertuples():
    recalculado = evaluate(y, probs[fila.modelo], fila.modelo)["log_loss"]
    assert abs(recalculado - fila.log_loss) < 1e-9, (fila.modelo, recalculado, fila.log_loss)
print("Las probabilidades recalculadas reproducen el CSV para los", len(registro), "modelos")

tabla = compare(registro.to_dict("records"))
tabla.round(4)

## 7. ¿Las diferencias son reales o ruido? Bootstrap pareado

Con 380 partidos, una diferencia de 0.005 en log loss puede ser suerte. Se remuestrean los partidos
**1000 veces** (semilla fija), siempre los mismos partidos para ambos modelos, y se calcula la diferencia
de log loss contra `logistica_sin_redundancia` (negativa = el otro modelo es mejor).
Si el intervalo del 95 % contiene el 0, **no se puede afirmar que haya diferencia**.
`prop_mejor` = fracción de remuestreos en que el otro modelo le gana a la logística.

In [ ]:
boot = tabla_bootstrap(y, probs, ref="logistica_sin_redundancia", n_boot=1000)
boot.to_csv("../docs/bootstrap_validacion.csv", index=False)
boot.round(4)

### ¿Y el modelo más simple?

`logistica_elo` (una sola variable) tampoco es significativamente peor que el modelo final en validación.
Como segunda opinión, comparamos las tres logísticas en la validación cruzada dentro de train
(mismos 5 pliegues temporales, 2017-18 a 2022-23).

In [ ]:
cv_log = {nombre: -tune_logistic(train, cols).best_score_
          for nombre, cols in {"logistica_elo": ["elo_diff"], "logistica_todas": FEATURE_COLS,
                               "logistica_sin_redundancia": FEATURES_FINAL}.items()}
(pd.DataFrame({"log_loss_cv_train": cv_log})
   .join(tabla.set_index("modelo")["log_loss"].rename("log_loss_val"))
   .sort_values("log_loss_cv_train").round(4))

## 8. Figura para la feria

In [ ]:
peores = set(boot.loc[boot["significativa"] & (boot["diferencia"] > 0), "modelo"]) - {"baseline_frecuencias"}
fig = fig_comparacion_modelos(tabla, "../docs/fig_comparacion_modelos.png", peores_signif=peores)

## 9. Conclusión

**¿Qué familia funciona mejor?** La **lineal**. Con las mismas variables, en validación (380 partidos):

| Familia | Mejor modelo | Log loss val | vs. `logistica_sin_redundancia` (IC 95 %) |
|---|---|---|---|
| Lineal | `logistica_sin_redundancia` | **0.9258** | (referencia) |
| Poisson | `poisson_elo` | 0.9305 | +0.0048 [−0.0036, +0.0131] → no significativa |
| Árboles | `xgboost` | 0.9339 | +0.0082 [−0.0042, +0.0221] → no significativa |
| Árboles | `random_forest` | 0.9411 | +0.0153 [+0.0020, +0.0281] → **significativamente peor** |

Los árboles también quedan detrás de la logística en la validación cruzada dentro de train (log loss CV:
logística 0.984, Random Forest 0.992, XGBoost 0.999), así que no es casualidad de la temporada 2023-24.

**¿Por qué los árboles no ganan?** Con pocas variables y una relación casi monótona (más Elo → más
probabilidad de ganar), la logística ya captura lo esencial con 33 parámetros. Los árboles cortan el Elo
en escalones y gastan su flexibilidad en encontrar patrones que no se repiten: en train ajustan mejor
(log loss 0.94 contra 0.97 de la logística) pero en partidos nuevos lo hacen peor. XGBoost eligió la
combinación **más frenada** de su rejilla (profundidad 2, tasa 0.03, 200 rondas): cuanto más libre, peor
(la más libre llega a 1.30 en CV). Sin frenos el sobreajuste es enorme: Random Forest sin límite de hojas
baja a 0.25 en train y sube a 0.97 en validación.

**¿Se mantiene `logistica_sin_redundancia` como modelo final?**

- **Frente a las familias nuevas, sí.** Ni los árboles ni Poisson la superan: ninguno tiene menor log loss
  en validación, y además son modelos más complejos (o, en el caso de Poisson, no mejores). No hay motivo
  para cambiar a árboles.
- **Pero el criterio, aplicado al pie de la letra, apunta a `logistica_elo`.** Su diferencia con el modelo
  final en validación (+0.0060, IC [−0.0012, +0.0136]) no es significativa, es mucho más simple (una sola
  variable) y en la validación cruzada dentro de train es incluso **mejor** (0.975 contra 0.984). Es decir:
  la forma, el descanso y los goles de los últimos 5 partidos no aportan una mejora demostrable sobre el Elo.
- **Recomendación:** según la regla acordada, el modelo final debería ser `logistica_elo`. El argumento para
  quedarse con `logistica_sin_redundancia` es práctico: tiene el mejor número en validación (gana en el 94 %
  de los remuestreos) y la app puede mostrar varios factores, no solo el Elo. **Es una decisión de equipo
  (A+B) antes del test del miércoles 21; en este PR no se cambia el modelo de la app.**

**Las cuotas del mercado** siguen por delante (0.9093, −0.0165 frente a nuestro modelo), pero el intervalo
[−0.0344, +0.0003] toca el 0: con 380 partidos ni siquiera la ventaja del mercado es concluyente al 95 %
(gana en el 97 % de los remuestreos). Es lo esperable: el mercado usa información que nosotros no tenemos
(lesiones, alineaciones, dinero de los apostadores).

**Limitación:** 380 partidos son pocos para separar modelos que difieren en milésimas. La comparación
definitiva será la corrida única en el test 2024-25 (miércoles 21), con el modelo que el equipo elija antes.
